In [1]:
import polars as pl

# Load lazily to reduce RAM usage
df = pl.read_parquet("../data/malaysia_transactions.parquet")

In [2]:
columns_to_keep = [
    "date_time",
    "ofi_entity_id",
    "rfi_entity_id",
    "trxn_amount",
    "trxn_type",
    "trxn_channel"
]

df_clean = df.select(columns_to_keep)

df_clean.null_count()

date_time,ofi_entity_id,rfi_entity_id,trxn_amount,trxn_type,trxn_channel
u32,u32,u32,u32,u32,u32
0,438281,464000,0,665434,0


In [3]:
# Drop any rows with missing sender/receiver IDs
df_filtered = df_clean.filter(
    pl.col("ofi_entity_id").is_not_null() & 
    pl.col("rfi_entity_id").is_not_null()
)

# Fill missing `trxn_type` with fallback label
df_filtered = df_filtered.with_columns(
    pl.col("trxn_type").fill_null("Unknown")
)

In [4]:
df_filtered.null_count()

date_time,ofi_entity_id,rfi_entity_id,trxn_amount,trxn_type,trxn_channel
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [5]:
df_filtered.shape

(11396551, 6)

In [6]:
# Sort the dataframe by time
df_sorted = df_filtered.sort("date_time")

# check earliest and latest timestamp
print("Start:", df_sorted["date_time"][0])
print("End:", df_sorted["date_time"][-1])

Start: 2025-06-01 00:00:00
End: 2025-06-30 23:59:59


In [7]:
from collections import defaultdict

graph = defaultdict(list)
incoming_graph = defaultdict(list)
entity_risk = defaultdict(float)
last_incoming_time = {}
last_outgoing_time = defaultdict(lambda: None)

In [8]:
from datetime import datetime, timedelta

# Rule threshold
MAX_LAYERING_DEPTH = 4
MAX_TIME_GAP = 600  # fast flow between nodes
defaultTimeGAP = 500

# Utility: parse timestamp if it's not already datetime
def ensure_datetime(ts):
    return ts if isinstance(ts, datetime) else datetime.strptime(str(ts), "%Y-%m-%d %H:%M:%S")

# Recursive DFS to measure path depth
def layering_depth(entity, visited, current_time, depth=0):
    if depth >= MAX_LAYERING_DEPTH:
        print(f"Max depth reached for {entity} at depth {depth}")
        return depth

    max_depth = depth
    for (target, t, amt, ttype) in graph[entity]:
        t = ensure_datetime(t)
        if target not in visited and (current_time - t) <= MAX_TIME_GAP:
            visited.add(target)
            new_depth = layering_depth(target, visited, t, depth + 1)
            max_depth = max(max_depth, new_depth)
            visited.remove(target)

    return max_depth

def process_transaction(sender, receiver, timestamp, amount, ttype):
    timestamp = ensure_datetime(timestamp)

    # Store in graphs
    graph[sender].append((receiver, timestamp, amount, ttype))
    incoming_graph[receiver].append((sender, timestamp, amount, ttype))

    ### --- Fan-in logic (receiver-based) ---
    prev_in_time = last_incoming_time.get(receiver)
    last_incoming_time[receiver] = timestamp  # update before using

    time_gap_in = (timestamp - prev_in_time).total_seconds() / 60 if prev_in_time else defaultTimeGAP
    risk_boost_in = max(0.0, (MAX_TIME_GAP - time_gap_in) / MAX_TIME_GAP)
    entity_risk[receiver] = 0.2 * entity_risk[receiver] + 0.8 * risk_boost_in

    ### --- Fan-out logic (sender-based) ---
    prev_out_time = last_outgoing_time.get(sender)
    last_outgoing_time[sender] = timestamp  # update before using

    time_gap_out = (timestamp - prev_out_time).total_seconds() / 60 if prev_out_time else defaultTimeGAP
    risk_boost_out = max(0.0, (MAX_TIME_GAP - time_gap_out) / MAX_TIME_GAP)
    entity_risk[sender] = 0.2 * entity_risk[sender] + 0.8 * risk_boost_out

    # Print both scores for debug
    # print(f"[IN ] Risk[{entity_risk[receiver]:.4f}] on receiver {receiver} — gap: {time_gap_in:.2f} min")
    # print(f"[OUT] Risk[{entity_risk[sender]:.4f}] on sender   {sender} — gap: {time_gap_out:.2f} min")

    # Block if either party too risky
    if entity_risk[receiver] >= 0.9:
        # print(f"⚠️ Receiver {receiver} risk too high: {entity_risk[receiver]:.2f}")
        return entity_risk[receiver]
    if entity_risk[sender] >= 0.9:
        # print(f"⚠️ Sender {sender} risk too high: {entity_risk[sender]:.2f}")
        return entity_risk[sender]

    return max(entity_risk[sender], entity_risk[receiver])



def layering_depth_backtrace(entity, visited, current_time, depth=0):
    if depth >= MAX_LAYERING_DEPTH:
        return depth

    max_depth = depth
    for (source, t, amt, ttype) in incoming_graph[entity]:
        t = ensure_datetime(t)
        if source not in visited and (current_time - t) <= MAX_TIME_GAP:
            visited.add(source)
            new_depth = layering_depth_backtrace(source, visited, t, depth + 1)
            max_depth = max(max_depth, new_depth)
            visited.remove(source)

    return max_depth

In [9]:
sucess_count = 0
fail_count = 0
for i in range(500000):  # or 1 million if fast enough
    row = df_sorted.row(i)
    return_value = process_transaction(
        sender=row[1],
        receiver=row[2],
        timestamp=row[0],
        amount=row[3],
        ttype=row[4]
    )
    if return_value < 0.8:
        sucess_count += 1
    else:
        fail_count += 1

print(f"sucess_count: {sucess_count}, fail_count: {fail_count}")

sucess_count: 413276, fail_count: 86724


MAX_TIME_GAP = 600, defaultTimeGAP = 500
sucess_count: 433327, fail_count: 66673 [0.9]
sucess_count: 413276, fail_count: 86724 [0.8]
[0.8~0.9] success_count: 20051


MAX_TIME_GAP = 10, defaultTimeGap = 8
sucess_count: 462534, fail_count: 37466

In [10]:
# len(entity_risk) == len([v for v in entity_risk.values() if v > 0])
len([v for v in entity_risk.values() if v > 0])

785175

785175

In [10]:
def print_top_risks(n=10):
    top = sorted(entity_risk.items(), key=lambda x: x[1], reverse=True)[:n]
    print("\n🔥 Top Risky Entities:")
    for eid, score in top:
        print(f"{eid}: {score:.4f}")

In [11]:
print_top_risks(25)


🔥 Top Risky Entities:
710112-02-6494: 0.9984
661124-02-7823: 0.9843
976578-I: 0.9808
900626-10-7219: 0.9792
915175-P: 0.9711
790908-07-6143: 0.9472
550718-10-7382: 0.9445
200607204345: 0.9419
671021-21-7041: 0.9386
790307-10-7796: 0.9380
500524-10-6442: 0.9365
952520-Y: 0.9352
SR7699043-N: 0.9336
647504-R: 0.9325
560607-01-7773: 0.9325
631130-49-6858: 0.9325
201605258766: 0.9285
940329-53-5679: 0.9272
589609-U: 0.9272
KD3512321-P: 0.9245
888577-G: 0.9219
TR8883602-W: 0.9205
200412171192: 0.9192
590804-10-5438: 0.9192
660828-08-6324: 0.9185


In [9]:
base_time = datetime(2025, 6, 1, 14, 0, 0)

synthetic_fan_out = [
    ("E500", "E601", base_time, 1000.0, "Online Transfer"),
    ("E500", "E602", base_time + timedelta(seconds=30), 980.0, "Online Transfer"),
    ("E500", "E603", base_time + timedelta(seconds=60), 970.0, "Online Transfer"),
    ("E500", "E604", base_time + timedelta(seconds=90), 960.0, "Online Transfer"),
    ("E500", "E605", base_time + timedelta(seconds=120), 950.0, "Online Transfer"),
]

for s, r, t, a, tt in synthetic_fan_out:
    risk = process_transaction(s, r, t, a, tt)
    print(f"Risk on {s} to {r}: {risk:.4f}")
    if risk >= 0.9:
        break

[IN ] Risk[0.1600] on receiver E601 — gap: 8.00 min
[OUT] Risk[0.1600] on sender   E500 — gap: 8.00 min
Risk on E500 to E601: 0.1600
[IN ] Risk[0.1600] on receiver E602 — gap: 8.00 min
[OUT] Risk[0.7920] on sender   E500 — gap: 0.50 min
Risk on E500 to E602: 0.7920
[IN ] Risk[0.1600] on receiver E603 — gap: 8.00 min
[OUT] Risk[0.9184] on sender   E500 — gap: 0.50 min
⚠️ Sender E500 risk too high: 0.92
Risk on E500 to E603: 0.9184


In [9]:
from datetime import datetime, timedelta

base_time = datetime(2025, 6, 1, 12, 0, 0)
synthetic_fan_in = [
    ("E001", "E100", base_time, 1000.0, "Online Transfer"),
    ("E002", "E100", base_time + timedelta(seconds=30), 1000.0, "Online Transfer"),
    ("E003", "E100", base_time + timedelta(seconds=30), 1000.0, "Online Transfer"),
]

for s, r, t, a, tt in synthetic_fan_in:
    return_risk = process_transaction(s, r, t, a, tt) 
    if return_risk < 0.9:
        print(f"return risk:{return_risk}\n")
    else:
        break

timestamp E001 - prev time None
Layering depth: 0, Time gap: 8.00 minutes
Risk[0.16000000000000003]: Processing transaction from E001 to E100 at 2025-06-01 12:00:00 with amount 1000.0
return risk:0.16000000000000003

timestamp E002 - prev time 2025-06-01 12:00:00
Layering depth: 0, Time gap: 0.50 minutes
Risk[0.792]: Processing transaction from E002 to E100 at 2025-06-01 12:00:30 with amount 1000.0
return risk:0.792

timestamp E003 - prev time 2025-06-01 12:00:30
Layering depth: 0, Time gap: 0.00 minutes
Risk[0.9584]: Processing transaction from E003 to E100 at 2025-06-01 12:00:30 with amount 1000.0
⚠️ Transaction blocked: Risk too high for E100 (0.96)


In [ ]:
# from datetime import datetime, timedelta

# base_time = datetime(2025, 6, 1, 12, 0, 0)

# for i in range(10):
#     s = f"E{i:03}"
#     r = f"E{i+1:03}"
#     t = base_time + timedelta(minutes=i)  # try shorter gap for first few
#     a = 1000.0 - i * 10
#     process_transaction(s, r, t, a, "Online Transfer")

In [ ]:
# print("Risk on E002:", entity_risk["E002"])
# print("Risk on E003:", entity_risk["E003"])
# print("Risk on E004:", entity_risk["E004"])
# print("Risk on E005:", entity_risk["E005"])

In [ ]:
# from datetime import datetime, timedelta

# base_time = datetime(2025, 6, 1, 12, 0, 0)

# synthetic_chain = [
#     ("E001", "E002", base_time, 1000.0, "Online Transfer"),
#     ("E002", "E003", base_time + timedelta(minutes=1), 980.0, "Online Transfer"),
#     ("E003", "E004", base_time + timedelta(minutes=2), 970.0, "Online Transfer"),
#     ("E004", "E005", base_time + timedelta(minutes=3), 950.0, "Online Transfer"),
# ]

# for s, r, t, a, tt in synthetic_chain:
#     process_transaction(s, r, t, a, tt)

# print("Risk on E002:", entity_risk["E002"])
# print("Risk on E003:", entity_risk["E003"])
# print("Risk on E004:", entity_risk["E004"])
# print("Risk on E005:", entity_risk["E005"])

In [9]:
# row = df_sorted.row(0)
# process_transaction(
#     sender=row[1],          # ofi_entity_id
#     receiver=row[2],        # rfi_entity_id
#     timestamp=row[0],       # date_time
#     amount=row[3],          # trxn_amount
#     ttype=row[4]            # trxn_type
# )

In [ ]:
# N = 100000  # start small, increase later if fast

# for i in range(N):
#     row = df_sorted.row(i)
#     process_transaction(
#         sender=row[1],
#         receiver=row[2],
#         timestamp=row[0],
#         amount=row[3],
#         ttype=row[4]
#     )

# # Inspect: how many entities have non-zero risk
# non_zero_risk = {k: v for k, v in entity_risk.items() if v > 0}
# print("Entities with non-zero risk:", len(non_zero_risk))

# # Show top 10 riskiest entities
# top_risky = sorted(non_zero_risk.items(), key=lambda x: x[1], reverse=True)[:10]
# print("Top risky entities:", top_risky)